# Macro Voting Timing Model — Consolidated Colab Notebook
Frozen spec: v3 (Python only, no R deliverable). This single notebook contains the
**entire pipeline inline** — data download/caching, factor transforms, z-scores, votes,
position rule, backtest engine, metrics, random-timing null, robustness suite, and all
6 required figures. No external repo/zip needed — just run every cell top to bottom.

**Requires a real internet connection to `fred.stlouisfed.org` and `finance.yahoo.com`**
— this is why it's meant for Colab (or any normal machine), not a locked-down sandbox.

**How to run:** Runtime -> Run all. First run takes a minute or two (downloads + caches
7 series to `data/raw/*.csv`); reruns reuse the cache automatically.

## 0. Setup

In [ ]:
!pip install -q pandas_datareader yfinance

import os, json, warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
%matplotlib inline

RAW_DIR = "data/raw"
FIG_DIR = "output/figures"
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

EVAL_START = "2000-01-01"
print("Setup OK.")

## 1. Data layer (spec Section 3)
Downloads/caches the 7 raw series, resamples everything to month-end, and applies the
**as-of availability lag** (Section 3.3): CFNAI and CPI are shifted forward one month at
load, before any transform, because those releases carry a real publication lag. Market-
price series (dollar, 2Y yield, S&P, HY OAS) need no shift — their month-end print is
already known by month-end.

In [ ]:
FRED_SERIES = {
    "CFNAI": "1985-01-01",
    "CPIAUCSL": "1985-01-01",
    "DTWEXBGS": "2006-01-01",       # inception; series itself begins Jan 2006
    "DGS2": "1985-01-01",
    "BAMLH0A0HYM2": "1996-12-01",   # inception; series begins Dec 1996
}
YAHOO_SERIES = {
    "^GSPC": "1985-01-01",      # price index -- SIGNAL (factor 5a)
    "^SP500TR": "1988-01-01",   # total return -- PERFORMANCE (Section 4)
}
STATISTICAL_RELEASE_SERIES = {"CFNAI", "CPIAUCSL"}  # get the +1 month as-of shift


def _cache_path(name):
    return os.path.join(RAW_DIR, f"{name.replace('^','')}.csv")


def _fetch_fred(series_id, start):
    import pandas_datareader.data as web
    df = web.DataReader(series_id, "fred", start=start, end=datetime.today())
    s = df[series_id].rename(series_id)
    s.index.name = "date"
    return s


def _fetch_yahoo(ticker, start):
    import yfinance as yf
    df = yf.download(ticker, start=start, interval="1mo", progress=False, auto_adjust=False)
    if df.empty:
        raise RuntimeError(f"yfinance returned no data for {ticker}")
    close = df["Close"]
    if isinstance(close, pd.DataFrame):
        close = close.iloc[:, 0]
    return close.rename(ticker).rename_axis("date")


def get_series(name, force_refresh=False):
    path = _cache_path(name)
    if os.path.exists(path) and not force_refresh:
        return pd.read_csv(path, index_col="date", parse_dates=True)[name]
    if name in FRED_SERIES:
        s = _fetch_fred(name, FRED_SERIES[name])
    elif name in YAHOO_SERIES:
        s = _fetch_yahoo(name, YAHOO_SERIES[name])
    else:
        raise KeyError(name)
    s.to_frame(name=name).rename_axis("date").to_csv(path)
    return s


def to_month_end(s):
    s = s.copy()
    s.index = pd.to_datetime(s.index)
    return s.resample("ME").last()


def build_raw_panel(force_refresh=False):
    raw = {name: get_series(name, force_refresh) for name in list(FRED_SERIES) + list(YAHOO_SERIES)}
    shifted = {}
    for name, s in raw.items():
        me = to_month_end(s)
        shifted[name] = me.shift(1) if name in STATISTICAL_RELEASE_SERIES else me
    panel = pd.DataFrame(shifted)
    panel.index.name = "date"
    return panel


panel = build_raw_panel(force_refresh=False)
print(panel.shape)
panel.tail()

## 2. Signal layer (spec Section 2)
Transform -> rolling 120-month z-score (min 60 months history, else abstain) -> signed
score (prior sign x z) -> vote ({-1,0,+1} at +-0.5 threshold) -> vote sum -> position rule
(sum>0 invested, sum<0 cash, sum==0 hold previous; first month defaults to invested).

In [ ]:
ROLL_WINDOW = 120
MIN_HISTORY = 60
VOTE_THRESHOLD = 0.5

SIGN_PRIOR = {
    "growth": +1, "inflation": -1, "trade": +1,
    "policy": -1, "risk_price": +1, "risk_spread": -1,
}


def transform_factors(panel):
    out = pd.DataFrame(index=panel.index)
    out["growth"] = panel["CFNAI"].rolling(3).mean()                      # CFNAI-MA3
    out["inflation"] = panel["CPIAUCSL"].pct_change(12) * 100             # CPI YoY
    out["trade"] = panel["DTWEXBGS"].pct_change(12) * 100                 # dollar 12M %chg
    out["policy"] = panel["DGS2"].diff(12)                                # 2Y 12M chg, pp
    out["risk_price"] = panel["^GSPC"].pct_change(12) * 100               # S&P 12M %chg
    out["risk_spread"] = panel["BAMLH0A0HYM2"].diff(12)                   # HY OAS 12M chg, pp
    return out


def rolling_zscores(transformed):
    z = pd.DataFrame(index=transformed.index)
    for col in transformed.columns:
        mean = transformed[col].rolling(ROLL_WINDOW, min_periods=MIN_HISTORY).mean()
        std = transformed[col].rolling(ROLL_WINDOW, min_periods=MIN_HISTORY).std()
        z[col] = (transformed[col] - mean) / std
    return z


def signed_scores(z):
    return pd.DataFrame({col: SIGN_PRIOR[col] * z[col] for col in z.columns}, index=z.index)


def _vote_from_score(s, threshold=VOTE_THRESHOLD):
    v = pd.Series(0, index=s.index, dtype=float)
    v[s > threshold] = 1
    v[s < -threshold] = -1
    v[s.isna()] = np.nan
    return v


def compute_votes(s, threshold=VOTE_THRESHOLD, exclude=None):
    votes = pd.DataFrame(index=s.index)
    for f in ("growth", "inflation", "trade", "policy"):
        if exclude != f:
            votes[f] = _vote_from_score(s[f], threshold)
    if exclude != "risk":
        rp, rs = s["risk_price"], s["risk_spread"]
        both, only_price = rp.notna() & rs.notna(), rp.notna() & rs.isna()
        risk_signed = pd.Series(np.nan, index=s.index)
        risk_signed[both] = (rp[both] + rs[both]) / 2
        risk_signed[only_price] = rp[only_price]
        votes["risk"] = _vote_from_score(risk_signed, threshold)
    return votes


def vote_sum(votes):
    return votes.sum(axis=1, skipna=True)


def compute_positions(vsum, start_invested=True):
    pos = pd.Series(index=vsum.index, dtype=float)
    prev = 1.0 if start_invested else 0.0
    for dt, v in vsum.items():
        cur = 1.0 if v > 0 else (0.0 if v < 0 else prev)
        pos[dt] = cur
        prev = cur
    return pos


def run_model(panel, threshold=VOTE_THRESHOLD, exclude=None, eval_start=EVAL_START):
    transformed = transform_factors(panel)
    z = rolling_zscores(transformed)
    s = signed_scores(z)
    votes = compute_votes(s, threshold=threshold, exclude=exclude)
    vsum = vote_sum(votes)
    positions = compute_positions(vsum[eval_start:], start_invested=True)
    return positions, votes, vsum, s


positions, votes, vsum, signed = run_model(panel)
print("Evaluation window:", positions.index.min().date(), "to", positions.index.max().date())
vsum.tail()

## 3. Backtest engine (spec Section 4)
The signal-to-return alignment shift lives HERE (not in the data layer) -- the end-of-
month-t signal earns month t+1's return, implemented as `positions.shift(1)`. Cash = 0%,
transaction costs = 0% (assignment constraints).

In [ ]:
def total_return_series(price_index):
    return price_index.pct_change()


def run_backtest(positions, sp500_tr_level):
    mkt_ret = total_return_series(sp500_tr_level)
    pos_aligned = positions.shift(1)                       # THE signal-to-return shift
    pos_aligned, mkt_ret_aligned = pos_aligned.align(mkt_ret, join="inner")
    strat_ret = pos_aligned * mkt_ret_aligned + (1 - pos_aligned) * 0.0
    df = pd.DataFrame({"position": pos_aligned, "mkt_return": mkt_ret_aligned,
                        "strategy_return": strat_ret}).dropna(subset=["position", "mkt_return"])
    df["strategy_equity"] = (1 + df["strategy_return"]).cumprod()
    df["bh_equity"] = (1 + df["mkt_return"]).cumprod()
    return df


def oracle_equity(sp500_tr_level, index_like):
    mkt_ret = total_return_series(sp500_tr_level).reindex(index_like.index)
    oracle_ret = pd.Series(np.where(mkt_ret > 0, mkt_ret, 0.0), index=mkt_ret.index)
    return (1 + oracle_ret).cumprod()


def switches_and_holding_period(position):
    pos = position.dropna()
    changes = max((pos != pos.shift(1)).sum() - 1, 0)
    n_spells = changes + 1
    return int(changes), float(len(pos) / n_spells)


bt_df = run_backtest(positions, panel["^SP500TR"])
n_switches, avg_holding = switches_and_holding_period(bt_df["position"])
print(f"Switches: {n_switches}, avg holding period: {avg_holding:.1f} months")
bt_df.tail()

## 4. Evaluation: metrics, confusion matrix, random-timing null, robustness (Section 5)

In [ ]:
PERIODS_PER_YEAR = 12

def cagr(returns):
    equity = (1 + returns).cumprod()
    n_years = len(returns) / PERIODS_PER_YEAR
    if n_years <= 0 or equity.iloc[-1] <= 0:
        return np.nan
    return equity.iloc[-1] ** (1 / n_years) - 1

def ann_vol(returns):
    return returns.std() * np.sqrt(PERIODS_PER_YEAR)

def sharpe(returns, rf=0.0):
    excess = returns - rf / PERIODS_PER_YEAR
    vol = excess.std()
    return np.nan if (vol == 0 or np.isnan(vol)) else (excess.mean() / vol) * np.sqrt(PERIODS_PER_YEAR)

def drawdown_series(returns):
    equity = (1 + returns).cumprod()
    return equity / equity.cummax() - 1

def max_drawdown(returns):
    return drawdown_series(returns).min()

def longest_drawdown_duration(returns):
    dd = drawdown_series(returns) < 0
    longest = cur = 0
    for flag in dd:
        cur = cur + 1 if flag else 0
        longest = max(longest, cur)
    return int(longest)

def calmar(cagr_v, maxdd_v):
    return np.nan if (maxdd_v == 0 or np.isnan(maxdd_v)) else cagr_v / abs(maxdd_v)

def hit_ratio(position, mkt_return):
    matched = ((position == 1) & (mkt_return > 0)) | ((position == 0) & (mkt_return <= 0))
    return matched.mean()

def capture_ratios(strategy_return, mkt_return):
    up, down = mkt_return > 0, mkt_return < 0
    comp = lambda s: (1 + s).prod() - 1
    up_strat, up_mkt = comp(strategy_return[up]), comp(mkt_return[up])
    down_strat, down_mkt = comp(strategy_return[down]), comp(mkt_return[down])
    return (up_strat/up_mkt if up_mkt else np.nan), (down_strat/down_mkt if down_mkt else np.nan)

def metrics_row(strategy_return, position, mkt_return, n_switches=None, avg_holding=None):
    c, dd = cagr(strategy_return), max_drawdown(strategy_return)
    up_cap, down_cap = capture_ratios(strategy_return, mkt_return)
    return {
        "CAGR": c, "ann_vol": ann_vol(strategy_return), "sharpe": sharpe(strategy_return),
        "max_drawdown": dd, "calmar": calmar(c, dd),
        "longest_dd_months": longest_drawdown_duration(strategy_return),
        "hit_ratio": hit_ratio(position, mkt_return) if position is not None else np.nan,
        "pct_months_in_market": position.mean() if position is not None else np.nan,
        "n_switches": n_switches, "avg_holding_period_months": avg_holding,
        "upside_capture": up_cap, "downside_capture": down_cap,
    }

def confusion_matrix(position, mkt_return):
    up, down = mkt_return > 0, mkt_return <= 0
    inv, cash = position == 1, position == 0
    return {"invested_and_up": int((inv & up).sum()), "invested_and_down": int((inv & down).sum()),
            "cash_and_up": int((cash & up).sum()), "cash_and_down": int((cash & down).sum()),
            "up_month_base_rate": float(up.mean())}

def random_timing_null(mkt_return, n_invested_months, n_sims=1000, seed=20000101):
    rng = np.random.default_rng(seed)
    n_total = len(mkt_return)
    sims = []
    for _ in range(n_sims):
        idx = rng.choice(n_total, size=n_invested_months, replace=False)
        pos = np.zeros(n_total); pos[idx] = 1.0
        sim_ret = pd.Series(pos, index=mkt_return.index) * mkt_return
        sims.append({"CAGR": cagr(sim_ret), "sharpe": sharpe(sim_ret)})
    return pd.DataFrame(sims)

def percentile_of(value, distribution):
    return float((distribution < value).mean() * 100)

DEFAULT_SUBPERIODS = {"2000-2009": ("2000-01-01", "2009-12-31"),
                       "2010-2019": ("2010-01-01", "2019-12-31"),
                       "2020-present": ("2020-01-01", None)}

def sub_period_metrics(strategy_return, position, mkt_return, bounds):
    rows = {}
    for label, (start, end) in bounds.items():
        sr, pos, mr = strategy_return[start:end], position[start:end], mkt_return[start:end]
        if len(sr):
            rows[label] = metrics_row(sr, pos, mr)
    return pd.DataFrame(rows).T

print("Evaluation functions defined.")

### 4a. Tripwire check (gate G8) -- read this before trusting anything below

In [ ]:
strategy_row = metrics_row(bt_df["strategy_return"], bt_df["position"], bt_df["mkt_return"], n_switches, avg_holding)
bh_row = metrics_row(bt_df["mkt_return"], pd.Series(1, index=bt_df.index), bt_df["mkt_return"], 0, float(len(bt_df)))

oracle_eq = oracle_equity(panel["^SP500TR"], bt_df)
oracle_ret = oracle_eq.pct_change().fillna(oracle_eq.iloc[0] - 1)
oracle_pos = (bt_df["mkt_return"] > 0).astype(int)
oracle_row = metrics_row(oracle_ret, oracle_pos, bt_df["mkt_return"],
                          int((oracle_pos != oracle_pos.shift(1)).sum()), np.nan)

null_df = random_timing_null(bt_df["mkt_return"], int(bt_df["position"].sum()), n_sims=1000, seed=20000101)
null_row = {"CAGR": null_df["CAGR"].mean(), "sharpe": null_df["sharpe"].mean()}

metrics_table = pd.DataFrame({"strategy": strategy_row, "buy_and_hold": bh_row,
                               "oracle": oracle_row, "random_null_mean": null_row}).T

strat_sharpe, strat_cagr, bh_cagr = strategy_row["sharpe"], strategy_row["CAGR"], bh_row["CAGR"]
tripped = bool(strat_sharpe > 1.0 or strat_cagr > bh_cagr + 0.05)
tripwire = {"tripped": tripped, "strategy_sharpe": float(strat_sharpe),
            "strategy_cagr": float(strat_cagr), "buy_hold_cagr": float(bh_cagr),
            "note": ("TRIPWIRE HIT -- halt and complete a look-ahead audit before presenting "
                     "any results (gate G8)." if tripped else
                     "Tripwire not triggered; no audit required.")}
print(tripwire["note"])
assert not tripwire["tripped"], "STOP: tripwire hit, audit before proceeding (see note above)."
tripwire

### 4b. Metrics table (Section 5.1)

In [ ]:
metrics_table

### 4c. Confusion matrix (Section 5.2)

In [ ]:
confusion = confusion_matrix(bt_df["position"], bt_df["mkt_return"])
confusion

### 4d. Random-timing null (Section 5.3)

In [ ]:
null_percentiles = {"CAGR_percentile": percentile_of(strategy_row["CAGR"], null_df["CAGR"]),
                    "sharpe_percentile": percentile_of(strategy_row["sharpe"], null_df["sharpe"])}
null_percentiles

### 4e. Robustness suite (Section 5.4): threshold sensitivity, sub-periods, leave-one-out ablation

In [ ]:
threshold_rows = {}
for th in (0.3, 0.5, 0.7):
    pos_th, *_ = run_model(panel, threshold=th)
    bdf = run_backtest(pos_th, panel["^SP500TR"])
    threshold_rows[f"threshold_{th}"] = metrics_row(bdf["strategy_return"], bdf["position"], bdf["mkt_return"])
threshold_table = pd.DataFrame(threshold_rows).T

subperiod_table = sub_period_metrics(bt_df["strategy_return"], bt_df["position"], bt_df["mkt_return"], DEFAULT_SUBPERIODS)

ablation_rows = {}
for factor in ("growth", "inflation", "trade", "policy", "risk"):
    pos_ab, *_ = run_model(panel, exclude=factor)
    bdf = run_backtest(pos_ab, panel["^SP500TR"])
    ablation_rows[f"without_{factor}"] = metrics_row(bdf["strategy_return"], bdf["position"], bdf["mkt_return"])
ablation_table = pd.DataFrame(ablation_rows).T

print("Threshold sensitivity:"); display(threshold_table)
print("\nSub-period metrics:"); display(subperiod_table)
print("\nLeave-one-out ablation:"); display(ablation_table)

## 5. Required figures

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(bt_df.index, bt_df["strategy_equity"], label="Strategy")
ax.plot(bt_df.index, bt_df["bh_equity"], label="Buy & Hold (TR)")
ax.plot(oracle_eq.index, oracle_eq.reindex(bt_df.index), label="Perfect-foresight oracle", linestyle="--")
ax.set_yscale("log"); ax.set_title("Fig 1. Growth of $1 (log scale)"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/01_growth_of_1.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(bt_df.index, drawdown_series(bt_df["strategy_return"]), label="Strategy")
ax.plot(bt_df.index, drawdown_series(bt_df["mkt_return"]), label="Buy & Hold")
ax.set_title("Fig 2. Drawdowns"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/02_drawdowns.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
sp = panel["^GSPC"].reindex(bt_df.index)
ax.plot(bt_df.index, sp, color="black", linewidth=1)
ax.fill_between(bt_df.index, sp.min(), sp.max(), where=bt_df["position"]==1, color="tab:green", alpha=0.15, label="Invested")
ax.fill_between(bt_df.index, sp.min(), sp.max(), where=bt_df["position"]==0, color="tab:red", alpha=0.15, label="Cash")
ax.set_title("Fig 3. Position ribbon over S&P 500 price index"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/03_position_ribbon.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
vsum_eval = vsum[EVAL_START:]
ax.plot(vsum_eval.index, vsum_eval, drawstyle="steps-post")
ax.axhline(0, color="black", linewidth=0.8)
for label, d in {"Risk(5b) spread joins": "2002-12-01", "Trade factor joins": "2012-01-01"}.items():
    ax.axvline(pd.Timestamp(d), color="grey", linestyle=":", alpha=0.7)
    ax.text(pd.Timestamp(d), ax.get_ylim()[1]*0.9, label, rotation=90, fontsize=7, va="top")
ax.set_title("Fig 4. Vote sum over time")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/04_vote_sum.png", dpi=140); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(null_df["sharpe"], bins=40, color="tab:blue", alpha=0.7)
ax.axvline(strategy_row["sharpe"], color="red", linewidth=2, label="Strategy Sharpe")
ax.set_title("Fig 5. Random-timing null: Sharpe distribution (n=1000)"); ax.legend()
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/05_random_null_sharpe.png", dpi=140); plt.show()

In [ ]:
signed_eval = signed[EVAL_START:]
fig, axes = plt.subplots(len(signed_eval.columns), 1, figsize=(10, 10), sharex=True)
for ax, col in zip(axes, signed_eval.columns):
    ax.plot(signed_eval.index, signed_eval[col])
    ax.axhline(0, color="black", linewidth=0.5)
    ax.set_ylabel(col, fontsize=8)
axes[0].set_title("Fig 6. Per-factor signed scores over time")
fig.tight_layout(); fig.savefig(f"{FIG_DIR}/06_factor_scores.png", dpi=140); plt.show()

## 6. Write output/metrics.json

In [ ]:
def _jsonify(obj):
    if isinstance(obj, (pd.DataFrame, pd.Series)):
        return json.loads(obj.to_json(orient="index"))
    if isinstance(obj, (np.floating, np.integer)):
        return obj.item()
    if isinstance(obj, dict):
        return {k: _jsonify(v) for k, v in obj.items()}
    return obj

payload = {
    "metrics_table": _jsonify(metrics_table),
    "confusion_matrix": _jsonify(confusion),
    "null_percentiles": _jsonify(null_percentiles),
    "threshold_table": _jsonify(threshold_table),
    "subperiod_table": _jsonify(subperiod_table),
    "ablation_table": _jsonify(ablation_table),
    "tripwire": _jsonify(tripwire),
    "n_switches": n_switches,
    "avg_holding_period_months": avg_holding,
    "evaluation_window": {"start": str(bt_df.index.min().date()), "end": str(bt_df.index.max().date()),
                           "n_months": int(len(bt_df))},
}
os.makedirs("output", exist_ok=True)
with open("output/metrics.json", "w") as f:
    json.dump(payload, f, indent=2, default=str)
print("Wrote output/metrics.json")

from google.colab import files
files.download("output/metrics.json")

## 7. Next step
Copy the printed tables above and the six figures in `output/figures/` into
`report/REPORT.md` Phase 5 (Performance) and Phase 6 (Interpretation for senior PMs).
Phases 1-4 (economic intuition, model design, skeptic's memo) don't depend on these
numbers and are already written in the full repo's `report/REPORT.md`.